# LLM-led autonomous nanoindentation on the Ta-Ti-Hf-Zr library

> **This notebook runs through the PyTango server.** It never drives the instrument directly.
> Every capture, stage move, indent and export is sent as a command to the PyTango (Tango
> Controls) device server running on the KLA iMicro control PC, which operates InView.
> Point `NANOINDENT_TANGO_HOST` at that PC's Tango database (`<host>:10000`), or set
> `USE_SIMULATOR = True` to run against the PyTango digital twin, which exposes the same
> devices and commands.

The operator states a goal in plain English; an orchestrated set of agents plans and runs
the campaign, with a Gaussian-process (GP) surrogate as the numerical advisor.

| phase | rounds | goal | what the GP offers |
|---|---|---|---|
| **coverage** | 1 to `COVERAGE_ROUNDS` | map the whole working area | unexplored points, weighted by GP uncertainty |
| **discovery** | the rest | highest hardness and modulus | Expected Improvement over the working area |

Each round the LLM site planner picks one move and gives its reason: `self` (it names the
coordinates), `consult_gp` (it reviews the GP's picks) or `delegate_gp` (the GP picks).
`CONTROL_MODE = "rules"` runs the earlier fixed workflow instead.

**Safety model**
1. Every approval gate ships empty (`MY_DECISION = ""`); approving is an explicit edit.
2. No environment variable can approve anything. Simulator auto-approval refuses to run on
   real hardware, and autonomous hardware runs need an acknowledgement typed in the session.
3. The LLM has no instrument access. Every site it names must pass the working-area check,
   the indent-spacing rule, the Measurement Strategy Agent validators and the approval gate.
4. Every gate, move, array, export and result is appended to a JSONL journal, so a campaign
   can be resumed after a kernel restart.

**Requirements:** the `nanoindentation_agents` package from this repository and a running
PyTango server (real or simulated). An LLM token is optional; without one, every agent uses
its deterministic fallback and the record says so.

## 1. Objective & switches

`USER_OBJECTIVE` is the only required input. Every other switch has a safe default; none of
them can place an indent by itself. `CONTROL_MODE`, `COVERAGE_ROUNDS` and `WORKING_AREA`
set who chooses the sites and where they may go.

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.max_colwidth", 90)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "nanoindentation_agents").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# ----------------------------- THE ONLY REQUIRED INPUT -----------------------------
USER_OBJECTIVE = os.environ.get("NB7_OBJECTIVE",
    "Map hardness and modulus across the Ta-Ti-Hf-Zr compositional gradient library, "
    "then discover the regions with the highest hardness and modulus. First cover the "
    "whole working area so no part of the composition gradient is missed; then "
    "concentrate the remaining rounds on and around the regions with the highest "
    "hardness and modulus, while still checking any unexplored region that could be "
    "better. Use one 2x2 array per site at 15 um spacing and 200 nm depth. Skip "
    "scratches, pores, particles and the sample edge. Keep measuring until the indent "
    "budget or round limit is reached. Report hardness and modulus maps of the "
    "library and rank the sites with the highest hardness and modulus.")



# ----------------------------- operator switches ------------------------------------
OPERATOR_NAME        = os.environ.get("NB7_OPERATOR", "Operator")   # recorded on every approval
USE_SIMULATOR        = False     # False -> real iMicro (PyTango server on the iMicro PC)
RUN_SIMULATOR_SERVER = False     # start the PyTango digital twin from this notebook
SKIP_SCREENING       = True     # True -> use the InView export (Process Review) as the only data source
# Seconds per indent. After the first result file appears, the runner waits
# (nx*ny - 1) * INDENT_DURATION_S before exporting. Raise it if arrays export early.
INDENT_DURATION_S    = 70.0
INDENT_BUDGET = int(os.environ.get("NB7_INDENT_BUDGET", "200"))  # rounds are the real budget
MAX_ROUNDS    = int(os.environ.get("NB7_MAX_ROUNDS", "10"))
SAMPLE_PREFIX = os.environ.get("NB7_SAMPLE_PREFIX", "test")        # -> test_01, test_02, ...

# ---------------------------- working area (Section 4B) -------------------------------
# Where indents may go, in camera-view axes (+x right, +y up, origin at the centre).
# The stage must be at `indenter_at` when Section 4B runs. Other options:
#   {"shape": "circle", "diameter_um": ...} | {"shape": "polygon", "vertices_um": [...]}
#   {"taught": "rectangle"}  (teach corners in Section 4B) | {"reuse_saved": True} | None
#   optional keys: "keepouts": [...], "travel_limits": (xmin, xmax, ymin, ymax)
WORKING_AREA = {"shape": "rectangle", "size_um": (5000, 5000),
                "indenter_at": "center", "edge_margin_um": 0}

# InView test folder; preflight checks it against the instrument before any indent.
TEST_NUMBER  = int(os.environ.get("NB7_TEST_NUMBER", "1"))
SESSION_DATE = os.environ.get("NB7_SESSION_DATE") or time.strftime("%Y%m%d")

# InView export format: "csv" or "excel" (.xlsx, needs openpyxl).
EXPORT_FORMAT = os.environ.get("NB7_EXPORT_FORMAT", "excel").strip().lower()

RUN_ID = os.environ.get("NB7_RUN_ID") or time.strftime("%m%d_%H%M%S")

from nanoindentation_agents.notebook_credentials import (
    load_project_env, resolve_hf_token)
load_project_env(REPO_ROOT)
# PyTango server: the Tango database on the iMicro PC (port 10000), or the local twin.
TANGO_HOST = os.environ.get(
    "NANOINDENT_TANGO_HOST", "localhost:10000" if USE_SIMULATOR else "YOUR_IMICRO_PC_IP:10000")
DISABLE_LLM      = os.environ.get("NB7_DISABLE_LLM", "0") == "1"
TRY_LLM          = bool(resolve_hf_token()) and not DISABLE_LLM
TRY_VISION_LLM   = TRY_LLM

# Per-site recipe advisor. Only features set to "agent" in MEASUREMENT_CONTROL_MODES
# below can be changed by it; "fixed" features keep their default value.
ENABLE_MEASUREMENT_STRATEGY_AGENT = True
POISSON      = 0.3
TARGET_DEPTH = 200   # nm
MIN_GRID     = 2     # smallest grid (nx, ny) allowed for a site

# ---- who chooses the sites (needs WORKING_AREA) -------------------------------------
# "llm"   -> LLM site planner; rounds 1..COVERAGE_ROUNDS map the area, later rounds discover.
# "rules" -> the earlier fixed workflow (COVERAGE_MODE / EXPLORE_PITCH_UM).
CONTROL_MODE     = os.environ.get("NB7_CONTROL_MODE", "llm").strip().lower()
COVERAGE_ROUNDS  = min(5, MAX_ROUNDS)
LLM_MAY_STOP     = False   # True -> the LLM may end discovery early

# "rules" mode only
COVERAGE_MODE    = True    # visit the least-explored point of the working area each round
EXPLORE_PITCH_UM = None    # coverage lattice pitch (um); None -> working-area size / 6
MEASUREMENT_CONTROL_MODES = {
    "site_action": "agent",       # indent or skip this proposed site
    "method_type": "fixed",      # keep the proven EH workflow
    "control_mode": "fixed",     # depth-controlled unless explicitly delegated
    "grid_nx": "agent",
    "grid_ny": "agent",
    "spacing_um": "agent",
    "rotation_deg": "fixed",
    "target depth": "fixed",
    "target load": "fixed",
    "strain rate": "fixed",
    "surface approach distance": "fixed",  # safety locked
    "surface approach velocity": "fixed",  # safety locked
    "poisson": "fixed",
    "metal": "fixed",
    "drift": "fixed",
}

print(f"Mode: {'SIMULATOR' if USE_SIMULATOR else 'REAL iMicro'} @ "
      f"tango://{TANGO_HOST} | run {RUN_ID} | "
      f"LLM {'on' if TRY_LLM else 'off (deterministic fallbacks)'}")
print(f"Budget: {INDENT_BUDGET} indents / {MAX_ROUNDS} adaptive rounds | operator: {OPERATOR_NAME}")
print(f"InView export format: {EXPORT_FORMAT.upper()}")
print(f"Measurement Strategy Agent: {'ENABLED' if ENABLE_MEASUREMENT_STRATEGY_AGENT else 'OFF (fixed/default fallback)'}")
assert CONTROL_MODE in ("llm", "rules"), f"CONTROL_MODE must be 'llm' or 'rules', not {CONTROL_MODE!r}"
print(f"Site control: {CONTROL_MODE.upper()}"
      + (f" - LLM leads ({COVERAGE_ROUNDS} coverage round(s), then discovery)"
         if CONTROL_MODE == "llm" else " - hard-coded workflow"))
print("OBJECTIVE:", USER_OBJECTIVE)


## 2. Connect & compose

One cell builds the whole stack: toolkit → session → journal → motion → exporter.
The **journal** is the durable spine - every gate, jog, array, export, and result
appends one JSONL line to `notebooks/campaign_journals/campaign_<RUN_ID>.jsonl`.


In [ ]:
import subprocess
from nanoindentation_agents import notebook_ui as ui
from nanoindentation_agents.state import create_initial_state
from nanoindentation_agents.config import load_config, set_active_config
from nanoindentation_agents.tools.indent_tools import IndentToolkit, set_toolkit
from nanoindentation_agents.instrument import (
    InstrumentSession, MotionController, ExportConfig, ExportPipeline,
    ScreenReadConfig, results as res)
from nanoindentation_agents.campaign import CampaignJournal

PYTANGO_DIR   = Path(os.environ.get("NB7_PYTANGO_DIR", REPO_ROOT.parent / "pytango based "))  # PyTango server code
DOWNLOAD_DIR  = REPO_ROOT / "notebooks" / "downloaded_data"
ANNOTATED_DIR = REPO_ROOT / "notebooks" / "annotated_images"
JOURNAL_DIR   = REPO_ROOT / "notebooks" / "campaign_journals"

sim_process = globals().get("sim_process")
if USE_SIMULATOR and RUN_SIMULATOR_SERVER:
    if sim_process is None or sim_process.poll() is not None:
        # The PyTango digital twin: same devices and commands as the iMicro.
        sim_process = subprocess.Popen(
            [sys.executable, "startup_scripts/run_servers.py",
             "--yaml", "configs/imicro-twin.yaml"],
            cwd=str(PYTANGO_DIR), stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
        time.sleep(15)   # database + device server start-up
        print(f"PyTango digital twin started (PID {sim_process.pid})")

cfg = load_config(None)
cfg.research_question = cfg.experiment_context = USER_OBJECTIVE
cfg.instrument.tango_host, cfg.instrument.sim_mode = TANGO_HOST, USE_SIMULATOR
cfg.literature.local_dir = str(REPO_ROOT / "literature")
cfg.memory.base_dir = str(REPO_ROOT / "experiment_memory")
cfg.measurement_strategy.enabled = ENABLE_MEASUREMENT_STRATEGY_AGENT
cfg.measurement_strategy.site_action.mode = MEASUREMENT_CONTROL_MODES["site_action"]
cfg.measurement_strategy.method_type.mode = MEASUREMENT_CONTROL_MODES["method_type"]
cfg.indent_defaults.method_control.control_mode_agent_selectable = (
    MEASUREMENT_CONTROL_MODES["control_mode"] == "agent")
for feature, spec in cfg.measurement_strategy.grid_control.params.items():
    spec.mode = MEASUREMENT_CONTROL_MODES[feature]
for feature, spec in cfg.indent_defaults.method_control.params.items():
    spec.mode = MEASUREMENT_CONTROL_MODES[feature]
cfg.indent_defaults.method_control.params["target depth"].default = TARGET_DEPTH
invalid_modes = {k: v for k, v in MEASUREMENT_CONTROL_MODES.items()
                 if v not in {"fixed", "agent"}}
assert not invalid_modes, f"Unknown control modes: {invalid_modes}"
set_active_config(cfg)

toolkit = cfg.instrument.make_toolkit(
    capture_dir=str(REPO_ROOT / "notebooks" / "downloaded_images"))
set_toolkit(toolkit)
# Raises with the real reason (database down, device not exported, firewall).
conn = toolkit.describe_connection()
print(f"Tango {conn['address']} | backend {conn.get('active_backend')} | "
      f"connected={conn.get('connected')} | {conn['latency_ms']} ms")
# Never drive the real stage while believing it is the twin, or vice versa.
assert bool(conn.get("simulated")) == USE_SIMULATOR, (
    f"USE_SIMULATOR={USE_SIMULATOR} but the Tango server reports "
    f"simulated={conn.get('simulated')} - fix TANGO_HOST / USE_SIMULATOR.")
toolkit.connect()   # attach the server to InView (no motion)

journal  = CampaignJournal(JOURNAL_DIR, RUN_ID)
session  = InstrumentSession(toolkit, download_dir=DOWNLOAD_DIR, run_id=RUN_ID,
                             journal=journal,
                             screen_config=ScreenReadConfig(csv_dir=str(DOWNLOAD_DIR)))
motion   = MotionController(session, use_simulator=USE_SIMULATOR,
                            calibration_path=JOURNAL_DIR / "stage_calibration.json")
exporter = ExportPipeline(session, ExportConfig(test_number=TEST_NUMBER,
                                                session_date=SESSION_DATE,
                                                export_format=EXPORT_FORMAT,
                                                use_simulator=USE_SIMULATOR),
                          download_dir=DOWNLOAD_DIR)

state = create_initial_state(research_question=USER_OBJECTIVE,
                             experiment_name="nb7_objective_campaign",
                             max_iterations=cfg.experiment.max_iterations,
                             ic_max_cycles=cfg.experiment.ic_max_cycles)
msg_log = ui.AgentMessageLog()
ui.ApprovalGate.registry.clear()

print(f"Connection OK | journal -> {journal.path}")
print(f"Motion: {motion.summary()}")


## 3. 🎯 Compile the objective - Gate 0

The orchestrator translates the free-text goal into a structured `ResearchObjective`
(LLM with a validated strict-JSON prompt when available, deterministic parser
otherwise). **You approve the machine's understanding before anything else runs.**


In [ ]:
from nanoindentation_agents.objective import compile_objective
from nanoindentation_agents.agents.orchestrator_agent import OrchestratorAgent

objective = compile_objective(USER_OBJECTIVE, cfg=cfg,
                              fov_bounds=(0.0, 100.0, 0.0, 75.0),  # re-locked after Observe
                              indent_budget=INDENT_BUDGET, max_rounds=MAX_ROUNDS,
                              use_llm=TRY_LLM)

# ---- campaign shaping: default 2x2 array at 15 um; success criteria fit MAX_ROUNDS ----
objective.constraints.grid_nx = objective.constraints.grid_ny = 2
objective.constraints.spacing_um = 15
objective.constraints.indent_budget = INDENT_BUDGET
objective.success.min_visited_locations = max(
    1, min(objective.success.min_visited_locations, MAX_ROUNDS - 1))
objective.success.min_confirming_arrays = max(
    1, min(objective.success.min_confirming_arrays, MAX_ROUNDS - 2))

orch = OrchestratorAgent(objective, cfg, use_llm=TRY_LLM)
state.update(orch.objective_state_updates())

display(pd.DataFrame([{"property": g.name, "direction": g.direction, "weight": g.weight,
                       "target range (GPa)": str(g.target_range) if g.target_range else "-"}
                      for g in objective.property_goals]))
print(f"Compiled by: {objective.compiled_by.upper()} | material: "
      f"{objective.material_system or '(not stated)'}")
print(f"Budget: {objective.constraints.indent_budget} indents | "
      f"{objective.constraints.max_rounds} rounds | "
      f"{objective.constraints.grid_nx}x{objective.constraints.grid_ny} arrays "
      f"@ {objective.constraints.spacing_um:g} um")
print(f"Success: >= {objective.success.min_improvement_pct:g}% over baseline, "
      f">= {objective.success.min_confirming_arrays} arrays, "
      f">= {objective.success.min_visited_locations} locations")

objective_gate = ui.ApprovalGate(
    "gate-0-objective", "Research objective approval",
    description=("Approve the orchestrator's interpretation of your research goal. "
                 "All briefings, success criteria and the budget derive from it.\n\n"
                 + objective.summary_text()))
msg_log.log("Orchestrator Agent", "Human Operator",
            f"Objective compiled ({objective.compiled_by}) - awaiting Gate 0",
            detail=objective.summary_text()[:300])


In [ ]:
# ==> YOUR DECISION IS REQUIRED HERE - the gate ships EMPTY on purpose. <==
#     MY_DECISION = "approve"            (optionally "approve: note")
#     MY_DECISION = "reject: <reason>"   (then refine USER_OBJECTIVE and re-run §3)
MY_DECISION = ""

from nanoindentation_agents.campaign import apply_decision
if apply_decision(objective_gate, MY_DECISION, OPERATOR_NAME):
    journal.append("GATE", gate_id="gate-0-objective",
                   approved=objective_gate.is_approved, by=OPERATOR_NAME)
    msg_log.log("Human Operator", "Orchestrator Agent",
                "Objective APPROVED - brief the sub-agents" if objective_gate.is_approved
                else "Objective REJECTED - refine USER_OBJECTIVE and re-run section 3")


In [ ]:
objective_gate.require_approved()
state.update(orch.briefing_state_updates(state))
display(pd.DataFrame([{"agent role": r, "orchestrator directive": t}
                      for r, t in state["orchestrator_briefings"].items()]))
print(f"Literature search pre-seeded: {state['literature_search_query']!r}")

from nanoindentation_agents.campaign import CampaignConfig, CampaignRunner
campaign = CampaignConfig(operator=OPERATOR_NAME, use_simulator=USE_SIMULATOR,
                          run_id=RUN_ID, sample_prefix=SAMPLE_PREFIX,
                          use_vision_llm=TRY_VISION_LLM, skip_screening=SKIP_SCREENING,
                          use_measurement_strategy_llm=TRY_LLM,
                          indent_duration_s=INDENT_DURATION_S,
                          annotated_dir=str(ANNOTATED_DIR))
# ---- who steers the indenter (Section 1) ----
campaign.site_control = CONTROL_MODE             # "llm" (default) | "rules"
campaign.coverage_rounds = COVERAGE_ROUNDS
campaign.use_site_llm = TRY_LLM                  # no LLM -> the GP chooses, and says so
campaign.llm_may_stop = LLM_MAY_STOP
campaign.coverage_mode = COVERAGE_MODE           # "rules" mode only
campaign.explore_pitch_um = EXPLORE_PITCH_UM
if CONTROL_MODE == "llm":
    print(f"site control: LLM-led - rounds 1-{COVERAGE_ROUNDS} coverage, then discovery; "
          f"LLM {'ON' if TRY_LLM else 'OFF -> the GP chooses every site'}; "
          f"LLM may stop early: {LLM_MAY_STOP}"
          f"{'' if WORKING_AREA else ' - needs WORKING_AREA, so the rules path will run'}")
else:
    print(f"site control: RULES - coverage mode {'ON' if COVERAGE_MODE else 'OFF'} "
          f"(grid pitch {EXPLORE_PITCH_UM or 'area/6'} um"
          f"{'' if WORKING_AREA else ' - needs WORKING_AREA, so it will not act'})")
runner = CampaignRunner(session=session, motion=motion, exporter=exporter,
                        journal=journal, objective=objective, orchestrator=orch,
                        state=state, app_cfg=cfg, campaign=campaign, msg_log=msg_log)
journal.append("CAMPAIGN_START", objective=USER_OBJECTIVE, operator=OPERATOR_NAME,
               simulator=USE_SIMULATOR, budget=INDENT_BUDGET, max_rounds=MAX_ROUNDS,
               measurement_strategy_enabled=ENABLE_MEASUREMENT_STRATEGY_AGENT,
               measurement_control_modes=MEASUREMENT_CONTROL_MODES)
print("Runner composed - the same propose/execute path serves manual AND autonomous rounds.")
print(f"Measurement Strategy Agent: {'ACTIVE' if ENABLE_MEASUREMENT_STRATEGY_AGENT and TRY_LLM else 'fixed/default fallback'}")
display(pd.DataFrame(runner.measurement_strategy.control_contract_rows()))
print(f"skip_screening: {SKIP_SCREENING} "
      f"({'Process Review/Excel is the sole data source' if SKIP_SCREENING else 'screen-read monitor active'})")
print(f"indent duration estimate: {INDENT_DURATION_S:.0f}s -> a 2x2 waits "
      f"{3 * INDENT_DURATION_S / 60:.1f} min after the result file appears")


## 4. 🔭 Observe, lock the search area, rank targets

The first capture fixes the campaign's coordinate frame. On real hardware the FOV is
**calibrated from the burned-in scale bar** (hard stop if unreadable - never move an
indenter on an unknown scale); the ranked targets seed round 1.


In [ ]:
from nanoindentation_agents.agents.observe_agent import observe_node

state.update(observe_node(state))
runner.lock_search_area()
ui.show_capture(state["current_image"], title="Initial video panel (search area)",
                note="This frame defines the campaign's search area.",
                fov_width_um=campaign.fov_w_um, fov_height_um=campaign.fov_h_um)

ranked = runner.analyze_current_frame(0, tag="initial", refresh_plan=True)
display(pd.DataFrame([{"rank": t["priority"], "target": t.get("label", "?"),
                       "x (um)": round(t.get("x_um", 0), 1),
                       "y (um)": round(t.get("y_um", 0), 1),
                       "objective score": t.get("objective_score"),
                       "why": t.get("rank_reason", "")}
                      for t in ranked[:6]]))


## 4B. 📐 Working area - the sample boundary

Set `WORKING_AREA` in Section 1. With one set, every site is checked against it: the **whole
array** must sit inside the dashed safe limit and clear of earlier indents, or the site is
**rejected, never moved**. Rounds refuse to run until you approve the map below.

**To teach the boundary**, jog with InView to each point, run the next cell with
`runner.teach_point()` uncommented, and repeat. Then run the cell after it.
- Rectangle: 2 opposite corners, or 3 corners in order for a rotated sample
- Circle: 3 points on the edge, well apart
- Polygon: 3 or more points in order around the outline


In [ ]:
# Jog to the point with InView first. This reads the position; it does not move anything.
# runner.teach_point()
print(f"Points taught so far: {getattr(runner, 'taught_points', [])}")


In [ ]:
from nanoindentation_agents.campaign import WorkingArea

WORKING_AREA_FILE = JOURNAL_DIR / "working_area.json"
area_gate = None
if WORKING_AREA is None:
    print("No working area set: no sample boundary in this run.")
else:
    if WORKING_AREA.get("reuse_saved"):
        area = WorkingArea.load(WORKING_AREA_FILE)
        print(f"Re-using the saved boundary: {WORKING_AREA_FILE}")
    elif WORKING_AREA.get("taught"):
        area = runner.area_from_taught_points(
            WORKING_AREA["taught"],
            edge_margin_um=WORKING_AREA.get("edge_margin_um", 100.0),
            travel_limits=WORKING_AREA.get("travel_limits"))
    else:
        area = runner.area_from_config(WORKING_AREA)
    area.save(WORKING_AREA_FILE)
    area_gate = runner.set_working_area(area)


In [ ]:
# ==> YOUR DECISION IS REQUIRED HERE when a working area is set <==
#     ("approve" / "reject: <reason>"). Check the map against the real sample first.
MY_DECISION = ""

if area_gate is not None and apply_decision(area_gate, MY_DECISION, OPERATOR_NAME):
    journal.append("GATE", gate_id=area_gate.gate_id, approved=area_gate.is_approved,
                   by=OPERATOR_NAME)


## 5. 📚💡 Context agents (optional) - Gate 1

Literature and hypothesis give the campaign scientific context and an auditable
prediction; they do not steer the acquisition function. Skip with
`RUN_CONTEXT_AGENTS = False` - the BO campaign is unaffected.


In [ ]:
RUN_CONTEXT_AGENTS = True

if RUN_CONTEXT_AGENTS:
    from nanoindentation_agents.agents.literature_agent import literature_node
    from nanoindentation_agents.agents.hypothesis_agent import hypothesis_node
    from nanoindentation_agents.critics.hypothesis_critic import hypothesis_critic_node

    lit = ui.run_with_timeout(lambda: literature_node(state), seconds=240,
                              label="Literature Agent")
    if lit and (lit.get("literature_context") or "").strip():
        state.update(lit)
    else:
        from nanoindentation_agents.rag.literature_retriever import LiteratureRetriever
        ctx, srcs = LiteratureRetriever(cfg.literature).retrieve(
            state["literature_search_query"],
            keywords=state.get("literature_search_keywords") or ["hardness", "modulus"])
        state.update({"literature_context": ctx[:2500], "literature_raw_context": ctx,
                      "literature_sources": srcs,
                      "literature_search_meta": {"mode": "local_files (offline fallback)"}})
    ui.literature_panel(state)

    hyp = ui.run_with_timeout(lambda: hypothesis_node(state), seconds=240,
                              label="Hypothesis Agent")
    if not hyp or not (hyp.get("hypothesis") or "").strip():
        goal = "; ".join(g.describe() for g in objective.property_goals)
        pct = objective.success.min_improvement_pct
        text = (f"We hypothesize that distinct microstructural regions of the "
                f"{objective.material_system or 'sample'} differ measurably in the "
                f"objective properties ({goal}); the objective is met if the best site "
                f"beats the sample baseline by >= {pct:g}% on the weighted score, and "
                f"spatially uniform properties refute it.")
        hyp = {"hypothesis": text,
               "hypothesis_history": state["hypothesis_history"] + [text],
               "hypothesis_data": {"hypothesis": text,
                                   "success_criteria": f">= {pct:g}% above baseline",
                                   "failure_criteria": "no site beats baseline in budget"}}
    state.update(hyp)
    ui.hypothesis_panel(state)
    for _ in range(3):
        state.update(hypothesis_critic_node(state))
        ui.critic_panel(f"Hypothesis Critic - pass {state['hypothesis_critic_passes']}",
                        state["hypothesis_critique"], state["hypothesis_critique_approved"])
        if (state["hypothesis_critique_approved"]
                and state["hypothesis_critic_passes"]
                >= cfg.experiment.min_hypothesis_critic_passes):
            break
        upd = ui.run_with_timeout(lambda: hypothesis_node(state), seconds=240,
                                  label="Hypothesis Agent (revision)")
        if upd:
            state.update(upd)
else:
    print("Context agents skipped.")

hypothesis_gate = ui.ApprovalGate(
    "gate-1-hypothesis", "Hypothesis / context sign-off",
    description="Approve to continue to calibration and rounds. No stage motion or "
                "indentation is possible until later gates are also approved.")


In [ ]:
# ==> YOUR DECISION IS REQUIRED HERE <==   ("approve" / "reject: <reason>")
MY_DECISION = ""

if apply_decision(hypothesis_gate, MY_DECISION, OPERATOR_NAME):
    state["user_approved"] = hypothesis_gate.is_approved
    journal.append("GATE", gate_id="gate-1-hypothesis",
                   approved=hypothesis_gate.is_approved, by=OPERATOR_NAME)


## 6. 🛠 Stage calibration & preflight

`motion.calibrate()` measures, per axis, the jog direction, gain and reversal cost
(6 small jogs, no indents, about 2 min) and saves it to
`campaign_journals/stage_calibration.json`. `runner.preflight()` refuses to start a
real-hardware campaign without a calibration newer than `campaign.calibration_max_age_h`
(default 12 h). The calibration first checks that the InView window is where the PyTango
server expects it, because clicks on a moved window land on the wrong control.

In [ ]:
hypothesis_gate.require_approved()

# CALIBRATION_MODE:
#   "auto"   -> probe the stage live (6 small jogs, no indents); refuses to save a
#               result that is inconsistent between repeat jogs
#   "manual" -> use the bench-test values below; jog_to() still stops if a live jog
#               moves the opposite way
#   "skip"   -> no calibration enforcement (troubleshooting only)
CALIBRATION_MODE = "skip"        # "auto" | "manual" | "skip"
RECALIBRATE      = True          # "auto" only: probe again even if a recent calibration exists
SETTLE_S         = 3.0           # wait after every move before reading the position back

# Bench-test values for "manual" mode
BENCH_DIR_X, BENCH_DIR_Y = -1, +1
BENCH_GAIN_X, BENCH_GAIN_Y = 1.0, 1.0

if not USE_SIMULATOR:
    motion.cfg.settle_s = SETTLE_S
    print(f"settle: waiting {motion.cfg.settle_s:g}s after every move before readback")

if USE_SIMULATOR:
    print("Simulator - ideal stage; calibration not required.")
elif CALIBRATION_MODE == "skip":
    campaign.require_motion_calibration = False
    print("WARNING: CALIBRATION_MODE='skip' - enforcement disabled. Using "
          f"whatever is already loaded: {motion.summary()}")
elif CALIBRATION_MODE == "manual":
    motion.pin_calibration(BENCH_DIR_X, BENCH_DIR_Y, BENCH_GAIN_X, BENCH_GAIN_Y,
                           source="isolated bench test 2026-08-03",
                           note="right/left/up/down 20 um each moved exactly 20 um "
                                "on one axis, 0.00 um cross-talk, fully reversible")
    journal.append("CALIBRATION", **{k: v for k, v in motion.calibration.items()
                                     if k != "axes"})
elif (not RECALIBRATE and motion.calibration
      and motion.calibration_age_h <= campaign.calibration_max_age_h):
    motion.apply_saved_calibration()
    print(f"Using saved calibration ({motion.calibration_age_h:.1f} h old): "
          f"{motion.summary()}")
    print("Set RECALIBRATE = True after anyone touches the InView window.")
else:
    # Raises StageSafetyError if a check fails: raise SETTLE_S, bring the InView
    # window to the front, or fall back to CALIBRATION_MODE = "manual".
    motion.calibrate()
    journal.append("CALIBRATION", **{k: v for k, v in motion.calibration.items()
                                     if k != "axes"})

# ---- image -> stage conversion (camera offset to stage move) ----------------------
# Re-measure after any objective, zoom or camera change.
CALIBRATE_IMAGE_TO_STAGE = False   # True -> measure now: XY only, no indents, about 1 min

if not USE_SIMULATOR:
    if CALIBRATE_IMAGE_TO_STAGE:
        runner.calibrate_image_to_stage(probe_um=60.0)
    else:
        print(f"image->stage: site_gain_x={campaign.site_gain_x:+.2f}, "
              f"site_gain_y={campaign.site_gain_y:+.2f}")
        print("  set CALIBRATE_IMAGE_TO_STAGE = True to re-measure on this field")


# ---- pre-indent arrival check (in image space) -----------------------------------
# Before indenting, the live frame is registered against the planning frame. If the
# chosen feature is not under the crosshair: correct once, re-check, then stop.
# A featureless field warns and proceeds.
VERIFY_ARRIVAL   = "enforce"   # "enforce" | "warn" | "off"   (never on the simulator)
ARRIVAL_TOL_UM   = 8.0         # how close the feature must be to the indenter axis
MAX_VISUAL_FIXES = 1           # residual corrections allowed before refusing

campaign.verify_arrival = VERIFY_ARRIVAL
campaign.arrival_tol_um = ARRIVAL_TOL_UM
campaign.max_visual_corrections = MAX_VISUAL_FIXES
if not USE_SIMULATOR:
    print(f"arrival check: {VERIFY_ARRIVAL} - tolerance {ARRIVAL_TOL_UM:g} um, "
          f"{MAX_VISUAL_FIXES} correction(s) allowed")
    print(f"  aiming at the burned-in indenter crosshair: {campaign.aim_at_crosshair}")

runner.preflight()              # ping + sitting folder + calibration enforcement
# Optional sanity check that the live screen keys match the recorder's expectations:
# session.screen_read_diagnostic()


## 7. 🎯 Rounds: one implementation, two calls

| call | what happens | instrument |
|---|---|---|
| `runner.propose_round()` | orchestrator decides, fresh frame and analysis, sites, Measurement Strategy Agent sets each recipe, gate opens | capture only |
| `runner.execute_round(p, decision=...)` | jog, **indent**, wait for the result file, export and download, validate, record and critique | **irreversible** |

Round 1 and every later round use the same path. Repeat the two cells below once per
round; re-running `propose_round()` only replaces an undecided gate. In LLM-led mode the
proposal shows the planner's move, reasoning and a GP map of the working area; each
decision is journaled as `LLM_SITE_CHOICE`. If the orchestrator returns `objective_met`,
`budget_exhausted` or `escalate_to_human`, go to Section 9.

In [ ]:
proposal = runner.propose_round()
if proposal.terminal:
    print(">> Campaign is over - skip to Section 9 (results).")
elif not proposal.plans:
    print(">> No viable sites this round - re-run propose_round() or widen the search.")
else:
    print(f">> Review the gate above, then set MY_DECISION in the next cell "
          f"(round {proposal.round_no}, {len(proposal.plans)} site(s)).")


In [ ]:
# ==> YOUR DECISION IS REQUIRED HERE - indentation is IRREVERSIBLE. <==
#     ("approve" / "reject: <reason>")
MY_DECISION = ""

round_results = runner.execute_round(proposal, decision=MY_DECISION)


## 8. 🤖 Autonomous campaign (explicit opt-in)

The same propose/execute path in a loop; only the approval policy changes.

* **Simulator:** `AutoApproval(use_simulator=True)` (refuses to run on hardware).
* **Real hardware:** `AutonomousHardwareApproval` needs the acknowledgement phrase typed in
  this session. No environment variable can enable it, and a bare Run All does nothing.

In [ ]:
from nanoindentation_agents.campaign import (AutoApproval, AutonomousHardwareApproval,
                                             AUTONOMOUS_ACK)

RUN_AUTONOMOUS = False   # set True AND construct a policy below to run hands-free

policy = None
if RUN_AUTONOMOUS and USE_SIMULATOR:
    policy = AutoApproval(use_simulator=True)
elif RUN_AUTONOMOUS:
    # Deliberate friction for real hardware - uncomment BOTH lines to accept that every
    # remaining round, including validated agent-selected recipes, places irreversible
    # indents with no per-round human gate:
    # I_UNDERSTAND = AUTONOMOUS_ACK
    # policy = AutonomousHardwareApproval(OPERATOR_NAME, ack=I_UNDERSTAND)
    pass

if policy is None:
    print("Autonomous mode not armed - use the manual round cells in Section 7.")
    if RUN_AUTONOMOUS and not USE_SIMULATOR:
        print("(Real hardware: uncomment the acknowledgement lines above to arm it.)")
else:
    stop_reason = runner.run(policy)
    print(f"Stop reason: {stop_reason}")


In [ ]:
# ---- Recovery: re-export any round that fell back to screen-read data --------------
# No indents, no motion; safe to re-run, also after a kernel restart.
if any(d.get("needs_recovery") for d in state["indent_history"]):
    runner.recover_pending(wait_s=60)
else:
    print("Nothing to recover: every round has validated export data.")

## 9. 📊 Results - maps, GP posterior, objective attainment


In [ ]:
all_rows = runner.results_rows()
print(f"{len(all_rows)} indents across "
      f"{all_rows['array'].nunique() if len(all_rows) else 0} arrays")
if len(all_rows):
    ui.plot_property_map(all_rows)
    summary = (all_rows.groupby("region")
               .agg(**{"n indents": ("Indent", "count"),
                       "H mean (GPa)": ("Hardness (GPa)", "mean"),
                       "H std": ("Hardness (GPa)", "std"),
                       "E mean (GPa)": ("Modulus (GPa)", "mean"),
                       "E std": ("Modulus (GPa)", "std")}).round(2))
    display(summary)


In [ ]:
# ============ Measured hardness & modulus maps (one marker per site = array mean) ============
# Reads recorded results only; no instrument calls.
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.transforms import Bbox

# ---- options --------------------------------------------------------------------
UNITS_MM          = True     # axes in mm from the first site (False -> absolute stage um)
MATCH_SAMPLE_VIEW = False    # True -> flip x so the map matches the camera / sample view
SHOW_AREA         = True     # dashed outline of the approved working area, if any
LABEL             = "short"  # "short" -> 01, 02 ... | "full" -> test_01 ...
CMAPS             = {"Hardness (GPa)": "viridis", "Modulus (GPa)": "viridis"}
FIG_W_IN          = 7.2      # double-column width; use 3.5 for single column
MARKER_S          = 70       # marker area (pt^2)

# ---- one row per site -------------------------------------------------------------
def _num(a):
    d = re.findall(r"\d+", str(a))
    return int(d[-1]) if d else 0

_rows = all_rows if "all_rows" in globals() else runner.results_rows()
sites = (_rows.groupby("array")
              .agg(x=("X (um)", "mean"), y=("Y (um)", "mean"),
                   H=("Hardness (GPa)", "mean"), H_sd=("Hardness (GPa)", "std"),
                   E=("Modulus (GPa)", "mean"), E_sd=("Modulus (GPa)", "std"),
                   n=("Hardness (GPa)", "size"))
              .reset_index()
              .sort_values("array", key=lambda s: s.map(_num))
              .reset_index(drop=True))
sites["label"] = sites["array"].map(
    lambda a: f"{_num(a):02d}" if LABEL == "short" and re.findall(r"\d+", str(a)) else str(a))
x0, y0 = (sites.loc[0, "x"], sites.loc[0, "y"]) if UNITS_MM else (0.0, 0.0)
scale = 1e-3 if UNITS_MM else 1.0
sites["px"] = (sites["x"] - x0) * scale
sites["py"] = (sites["y"] - y0) * scale


def place_labels(ax, xs, ys, texts, *, marker_s, fontsize):
    """Greedy, collision-free label placement around each marker."""
    fig = ax.figure
    fig.canvas.draw()
    pts = ax.transData.transform(np.column_stack([xs, ys]))
    r_px = 0.5 * np.sqrt(marker_s) * fig.dpi / 72.0 + 1.5        # marker radius + edge
    taken = [Bbox.from_extents(px - r_px, py - r_px, px + r_px, py + r_px) for px, py in pts]
    inside = ax.get_window_extent()
    dirs = [(1, 0), (-1, 0), (0, 1), (0, -1), (1, 1), (-1, 1), (1, -1), (-1, -1)]
    base = 0.5 * np.sqrt(marker_s) + 2.0                           # just clear of the marker (pt)
    # crowded sites first, so they get the best spots
    d = np.hypot(pts[:, None, 0] - pts[None, :, 0], pts[:, None, 1] - pts[None, :, 1])
    order = np.argsort(np.sort(d, axis=1)[:, 1] if len(pts) > 1 else [0])
    for i in order:
        placed = None
        for step in (1.0, 2.2, 3.6, 5.0):
            for dx, dy in dirs:
                off = (dx * base * step / np.hypot(dx, dy), dy * base * step / np.hypot(dx, dy))
                ha = "left" if dx > 0 else "right" if dx < 0 else "center"
                va = "bottom" if dy > 0 else "top" if dy < 0 else "center"
                t = ax.annotate(texts[i], (xs[i], ys[i]), xytext=off, textcoords="offset points",
                                ha=ha, va=va, fontsize=fontsize, color="0.15", zorder=6)
                bb = t.get_window_extent(fig.canvas.get_renderer()).expanded(1.08, 1.15)
                ok = (inside.contains(bb.x0, bb.y0) and inside.contains(bb.x1, bb.y1)
                      and not any(bb.overlaps(o) for o in taken))
                if ok:
                    placed = (t, bb, step)
                    break
                t.remove()
            if placed:
                break
        if placed is None:                     # nothing free: put it out right, with a leader
            t = ax.annotate(texts[i], (xs[i], ys[i]), xytext=(base * 5, 0),
                            textcoords="offset points", ha="left", va="center",
                            fontsize=fontsize, color="0.15", zorder=6)
            placed = (t, t.get_window_extent(fig.canvas.get_renderer()), 5.0)
        t, bb, step = placed
        taken.append(bb)
        if step > 1.0:                          # moved away from its dot: add a leader line
            kw = dict(xytext=t.xyann, textcoords="offset points", ha=t.get_ha(),
                      va=t.get_va(), fontsize=fontsize, color="0.15", zorder=6)
            t.remove()
            ax.annotate(texts[i], (xs[i], ys[i]), **kw,
                        arrowprops=dict(arrowstyle="-", lw=0.5, color="0.45", shrinkA=1.0,
                                        shrinkB=0.5 * np.sqrt(marker_s) + 0.5))


style = {"font.family": "sans-serif",
         "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
         "font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8,
         "xtick.labelsize": 7, "ytick.labelsize": 7, "axes.linewidth": 0.6,
         "xtick.major.width": 0.6, "ytick.major.width": 0.6,
         "savefig.dpi": 600, "pdf.fonttype": 42, "svg.fonttype": "none"}
with plt.rc_context(style):
    fig, axes = plt.subplots(1, 2, figsize=(FIG_W_IN, FIG_W_IN * 0.46), layout="constrained")
    span_x = np.ptp(sites["px"]) or 1.0
    span_y = np.ptp(sites["py"]) or 1.0
    pad = 0.12 * max(span_x, span_y)
    for ax, (col, key, name, sym), letter in zip(
            axes, [("Hardness (GPa)", "H", "Hardness", "H"),
                   ("Modulus (GPa)", "E", "Elastic modulus", "E")], "ab"):
        if SHOW_AREA and getattr(runner, "working_area", None) is not None:
            o = runner.working_area.boundary.outline()
            ax.plot((o[:, 0] - x0) * scale, (o[:, 1] - y0) * scale, ls="--", lw=0.6,
                    color="0.6", zorder=1)
        sc = ax.scatter(sites["px"], sites["py"], c=sites[key], cmap=CMAPS[col], s=MARKER_S,
                        edgecolor="k", linewidths=0.5, zorder=4)
        cb = fig.colorbar(sc, ax=ax, shrink=0.9, pad=0.02)
        cb.set_label(f"{name}, {sym} (GPa)")
        cb.outline.set_linewidth(0.6)
        cb.ax.tick_params(width=0.6, labelsize=7)
        ax.set_xlim(sites["px"].min() - pad, sites["px"].max() + pad)
        ax.set_ylim(sites["py"].min() - pad, sites["py"].max() + pad)
        ax.set_aspect("equal")
        if MATCH_SAMPLE_VIEW:
            ax.invert_xaxis()
        unit = "mm" if UNITS_MM else "µm"
        ax.set_xlabel(f"{'Δx' if UNITS_MM else 'Stage x'} ({unit})")
        ax.set_ylabel(f"{'Δy' if UNITS_MM else 'Stage y'} ({unit})")
        ax.grid(alpha=0.15, lw=0.5)
        ax.text(-0.02, 1.02, f"({letter})", transform=ax.transAxes, fontsize=10,
                fontweight="bold", ha="right", va="bottom")
    fig.canvas.draw()                                   # settle the layout first
    fig.set_layout_engine("none")                       # freeze it, then place labels
    for ax in axes:
        place_labels(ax, sites["px"].to_numpy(), sites["py"].to_numpy(),
                     sites["label"].tolist(), marker_s=MARKER_S, fontsize=7)

    out_dir = REPO_ROOT / "paper" / "figures" / RUN_ID
    out_dir.mkdir(parents=True, exist_ok=True)
    for ext in ("pdf", "svg", "png"):
        fig.savefig(out_dir / f"property_map.{ext}", bbox_inches="tight")
    plt.show()

print(f"Saved: {out_dir / 'property_map.pdf'} (+ .svg, .png)")
display(sites[["array", "x", "y", "n", "H", "H_sd", "E", "E_sd"]].round(2))


In [ ]:
import matplotlib.pyplot as plt
from nanoindentation_agents.analysis.objective_acquisition import ObjectiveSurrogate

final_surrogate = ObjectiveSurrogate(objective).fit(state["indent_history"])
final_progress = orch.evaluate_progress(state, final_surrogate)

if final_surrogate.n_points == 0:
    print("NOTE: the surrogate has NO observations - the panels below are the "
          "untrained prior. Fix the data first (recover_pending).")
# Map the GP over the whole working area when there is one (not just the first camera field).
_area_bounds = None
if runner.working_area is not None:
    _o = runner.working_area.boundary.outline()
    _area_bounds = (_o[:, 0].min(), _o[:, 0].max(), _o[:, 1].min(), _o[:, 1].max())
maps = final_surrogate.posterior_maps(n=70, bounds=_area_bounds)
best = final_surrogate.best_observed()
fig, axes = plt.subplots(2, 2, figsize=(12.5, 9))
for ax, (key, title, cmap) in zip(axes.ravel(), [
        ("hardness_mean", "predicted hardness (GPa)", "inferno"),
        ("modulus_mean", "predicted modulus (GPa)", "viridis"),
        ("scalar_mean", "objective score (weighted)", "cividis"),
        ("scalar_sigma", "objective uncertainty (sigma)", "magma")]):
    im = ax.imshow(maps[key], origin="lower", extent=maps["extent"],
                   aspect="equal", cmap=cmap)
    ax.scatter(final_surrogate.X[:, 0], final_surrogate.X[:, 1], s=12, c="white",
               edgecolor="k", linewidths=0.4)
    if best:
        ax.scatter([best["x_um"]], [best["y_um"]], marker="*", s=340, c="red",
                   edgecolor="k", linewidths=0.7)
    ax.set_title(title); ax.set_xlabel("stage x (um)"); ax.set_ylabel("stage y (um)")
    fig.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle(f"Final GP posterior ({final_surrogate.mode}, "
             f"{final_surrogate.n_points} indents) - star = best site", y=1.0)
plt.tight_layout(); plt.show()

display(final_progress.table())
if best:
    print(f"Best site: ({best['x_um']:.1f}, {best['y_um']:.1f}) um - "
          f"H={best['hardness_gpa']:.2f} GPa, E={best['modulus_gpa']:.1f} GPa")
print(f"Improvement over baseline: {final_progress.improvement_pct:.1f}% "
      f"(required {objective.success.min_improvement_pct:g}%)")
verdict = "OBJECTIVE MET" if final_progress.all_criteria_met else "OBJECTIVE NOT (YET) MET"
print(f"\n{'=' * 72}\n{verdict}\n{'=' * 72}")


In [ ]:
# ---- How each round's site was chosen (LLM-led mode) --------------------------------
# Reads the recorded decisions only: no instrument calls. Empty in "rules" mode.
_choices = state.get("llm_site_choices") or journal.events("LLM_SITE_CHOICE")
if not _choices:
    print("No LLM site decisions recorded (rules mode, or no LLM-led round ran).")
else:
    display(pd.DataFrame([{
        "round": ch["round_no"], "phase": ch["phase"], "LLM mode": ch["mode"],
        "decided by": ch["advisor"],
        "GP verdict": ch.get("gp_verdict") or "-",
        "site(s)": "; ".join(f"({s['x_um']:.0f}, {s['y_um']:.0f}) {s['origin']}"
                            for s in ch["sites"]),
        "GP #1": (f"({ch['gp_recommendations'][0]['x_um']:.0f}, "
                  f"{ch['gp_recommendations'][0]['y_um']:.0f})"
                  if ch["gp_recommendations"] else "-"),
        "rejected": len(ch.get("rejected") or []),
        "reasoning": ch["reasoning"][:140],
    } for ch in _choices]))
    _modes = pd.Series([ch["mode"] for ch in _choices]).value_counts()
    print("LLM modes used:", ", ".join(f"{m} x{n}" for m, n in _modes.items()))


## 9B. 🧑‍🔬 Final Analysis Agent

Runs once after the rounds. The numbers are computed deterministically from the recorded
indents; the LLM only writes the narrative around them (findings, comparison, conclusion,
limitations, next steps). No instrument calls; safe to re-run.

By default it refuses to run while any round is still on screen-read data. Run the
recovery cell first, or set `REQUIRE_VALIDATED_DATA = False`, in which case those rounds
are listed as provisional in the report.

In [ ]:
# ============ Final Analysis Agent (reads recorded data only) ============
from collections import Counter
from nanoindentation_agents.agents.final_analysis_agent import FinalAnalysisAgent

# Optional reference hardness bands (GPa), keyed by region label. Leave empty if none.
LIT_EXPECTATIONS = globals().get("LIT_EXPECTATIONS", {})
REQUIRE_VALIDATED_DATA = True   # refuse to analyse rounds still on screen-read data

all_rows = runner.results_rows()
pending  = [d["sample_name"] for d in state["indent_history"] if d.get("needs_recovery")]
print("data source per round:",
      dict(Counter(d.get("data_source", "unknown") for d in state["indent_history"])))

if len(all_rows) == 0:
    print("No indents recorded - nothing to analyze.")
elif pending and REQUIRE_VALIDATED_DATA:
    print(f"BLOCKED: {len(pending)} round(s) still on screen data: {pending}\n"
          "  -> run the recovery cell (runner.recover_pending(); no new indents),\n"
          "     then re-run this cell. Set REQUIRE_VALIDATED_DATA = False to analyze anyway.")
else:
    ui.agent_banner("Final Analysis Agent",
                    "reviewing every completed test like a nanoindentation expert")

    # Section 9 computes final_progress; recompute it if that cell was skipped.
    final_progress = globals().get("final_progress")
    if final_progress is None:
        from nanoindentation_agents.analysis.objective_acquisition import ObjectiveSurrogate
        final_progress = orch.evaluate_progress(
            state, ObjectiveSurrogate(objective).fit(state["indent_history"]))

    analyst = FinalAnalysisAgent(objective, cfg, use_llm=TRY_LLM)
    report  = analyst.analyze(all_rows, state=state,
                              expectations=LIT_EXPECTATIONS, progress=final_progress)

    if pending:   # only reachable with REQUIRE_VALIDATED_DATA = False - say so in the report
        note = (f"{len(pending)} round(s) were read off the instrument screen rather than a "
                f"validated export file ({', '.join(pending)}) - treat those numbers as "
                f"provisional.")
        report.limitations = f"{report.limitations}\n\n{note}".strip()
        report.flags.append(note)

    report.display()

    # Persist it where this notebook keeps the rest of its evidence.
    state["final_analysis"] = report.to_dict()
    journal.append("FINAL_ANALYSIS", advisor=report.advisor, n_indents=report.n_indents,
                   n_tests=report.n_tests, headline=report.headline, flags=report.flags,
                   conclusion=report.conclusion[:600])
    md_path = REPO_ROOT / "experiment_logs" / f"nb7_final_analysis_{RUN_ID}.md"
    md_path.parent.mkdir(parents=True, exist_ok=True)
    md_path.write_text(report.to_markdown(), encoding="utf-8")
    msg_log.log("Final Analysis Agent", "Orchestrator Agent",
                "Campaign review complete - findings, conclusion, limitations, next steps.",
                detail=report.conclusion[:280])
    print(f"\nNarrative by: {report.advisor.upper()}  |  saved: {md_path}")


## 10. 🧾 Run report & audit

The journal is the audit trail. The JSON report adds the compiled objective, decisions,
gate records and the per-feature measurement-strategy history.

In [ ]:
print(json.dumps(journal.summary(), indent=2))
display(journal.measurement_strategy_history())
display(journal.measurement_strategy_feature_history())
display(journal.method_plan_history())
display(ui.ApprovalGate.audit_table())
msg_log.render_transcript()

report_path = ui.save_run_report(
    REPO_ROOT / "experiment_logs" / f"nb7_campaign_{RUN_ID}.json",
    operator=OPERATOR_NAME, user_objective=USER_OBJECTIVE,
    objective=state.get("research_objective"),
    orchestrator_briefings=state.get("orchestrator_briefings"),
    orchestrator_decisions=state.get("orchestrator_decisions", []),
    measurement_strategy_enabled=ENABLE_MEASUREMENT_STRATEGY_AGENT,
    measurement_control_modes=MEASUREMENT_CONTROL_MODES,
    measurement_strategy_decisions=state.get("measurement_strategy_decisions", []),
    objective_progress=final_progress.to_dict(),
    acquisition=state.get("acquisition_meta", {}),
    hypothesis=state.get("hypothesis"),
    literature={"query": state.get("literature_search_query"),
                "sources": state.get("literature_sources")},
    indent_arrays=[{k: v for k, v in d.items() if k != "results"}
                   for d in state["indent_history"]],
    measurement_critique=state.get("measurement_critique"),
    final_decision=state.get("measurement_critique_decision"),
    final_analysis=state.get("final_analysis"),   # from Section 9B, if it ran
    approval_gates=[g.record() for g in ui.ApprovalGate.registry],
    agent_messages=msg_log.entries,
    campaign_journal=str(journal.path))
print("Report saved:", report_path)


## 11. ♻️ Resume after a kernel restart / shutdown

The journal makes a crash recoverable: re-run sections **1–3** (same `TEST_NUMBER`;
approve Gate 0), then point a journal at the old run id and rebuild. Physical indents
whose exports never validated are then re-exported with **no new indents**.

```python
journal = CampaignJournal(JOURNAL_DIR, "<old RUN_ID>")     # see the table below
# rebuild session/runner against that journal (re-run the compose + runner cells),
runner.resume_from_journal()      # indent history + budget + GP observations
runner.recover_pending()          # re-export anything still on screen data
```


In [ ]:
runs = CampaignJournal.list_runs(JOURNAL_DIR)
display(pd.DataFrame(runs)) if runs else print("No journaled runs yet.")

if sim_process is not None and sim_process.poll() is None:
    sim_process.terminate()
    try:
        sim_process.wait(timeout=5)
    except Exception:
        sim_process.kill()
    print("Simulator server stopped.")


## 12. 📄Figures

Generates the standard figure set from the finished campaign (sites, H and E per test,
load-displacement curves, depth profiles, GP posterior, improvement and EI by round,
H against E, arrival accuracy, round timing). Run it after recovery so nothing is plotted
from screen-read numbers. It reads the journal, the downloaded exports and the saved
frames only. Figures with missing inputs are skipped and listed in
`figures_manifest.json`; draft captions are written to `captions.md`.

To re-plot an earlier run without notebook state:

```python
CampaignFigures.from_journal(JOURNAL_DIR, "<RUN_ID>",
                             download_dir=DOWNLOAD_DIR).generate_all(out)
```

In [ ]:
# ============ Publication figures (reads recorded data only) ============
from nanoindentation_agents.analysis.publication import CampaignFigures, PubStyle

FIG_DIR = REPO_ROOT / "paper" / "figures" / RUN_ID
FORMATS = ("pdf", "png")          # PDF (vector, TrueType) for the journal; PNG for talks
FIG_STYLE = PubStyle(base_pt=8, dpi=600)

# Optional micrograph for Figure 1a, with a (left, top, right, bottom) pixel crop.
MICROGRAPH = None
MICROGRAPH_CROP = None

# Refit the surrogate if Section 9 was skipped.
if "final_surrogate" not in globals():
    from nanoindentation_agents.analysis.objective_acquisition import ObjectiveSurrogate
    final_surrogate = ObjectiveSurrogate(objective).fit(state["indent_history"])
    final_progress = orch.evaluate_progress(state, final_surrogate)

pending = [d["sample_name"] for d in state["indent_history"] if d.get("needs_recovery")]
if pending:
    print(f"NOTE: {len(pending)} round(s) still on screen-read data {pending} - those "
          "points are plotted from OCR, not from a validated export. Run "
          "runner.recover_pending() first if you can.\n")

figs = CampaignFigures.from_runner(runner, state=state, surrogate=final_surrogate,
                                   progress=final_progress, style=FIG_STYLE)
figs.micrograph, figs.micrograph_crop = MICROGRAPH, MICROGRAPH_CROP
manifest = figs.generate_all(FIG_DIR, formats=FORMATS)


In [ ]:
# ---- Review the set, with its draft captions ---------------------------------------
from IPython.display import Image as IPyImage, Markdown

display(Markdown((FIG_DIR / "captions.md").read_text(encoding="utf-8")))

for f in manifest["figures"]:
    png = next((p for p in f["files"] if p.endswith(".png")), None)
    if png:
        display(Markdown(f"---\n**Figure {f['n']} - {f['title']}**"))
        display(IPyImage(filename=png, width=620))
    else:
        display(Markdown(f"---\n**Figure {f['n']} - {f['title']}**  \n"
                         f"_skipped: {f.get('skipped')}_"))

print(f"\nVector PDFs and 600 dpi PNGs: {FIG_DIR}")
print("Drop the PDFs straight into LaTeX; captions.md is a starting point, not a "
      "finished caption - check every number in it before submission.")


In [ ]:
# ============ Inspect the downloaded InView exports for this run (CSV or Excel) ============
# Reads local files only; safe to re-run.
import re
from pathlib import Path
import pandas as pd
from IPython.display import display
from nanoindentation_agents.instrument import results as R

RUN_ID       = globals().get("RUN_ID", "YOUR_RUN_ID")
DOWNLOAD_DIR = Path(globals().get("DOWNLOAD_DIR", REPO_ROOT / "notebooks" / "downloaded_data"))
SHOW_RAW     = False    # True -> print each Results table verbatim
SHOW_CURVES  = True     # True -> one summary row per indent curve

EXPORT_EXTS = {".csv", *R.EXCEL_EXTS}

pd.set_option("display.width", 200)


def run_exports() -> list:
    """One record per test of this run: {test, files, summary, curves}."""
    out = {}
    for p in sorted(DOWNLOAD_DIR.iterdir()):
        if not p.is_file() or p.suffix.lower() not in EXPORT_EXTS:
            continue
        m = re.match(rf"(.+?)_{RUN_ID}_export(?:_(.+))?$", p.stem, re.I)
        if not m:
            continue
        test, kind = m.group(1), (m.group(2) or "")
        rec = out.setdefault(test, {"test": test, "files": [], "fmt": set(),
                                    "summary": None, "curves": {}})
        rec["files"].append(p)
        rec["fmt"].add(p.suffix.lstrip(".").lower())
        try:
            if p.suffix.lower() in R.EXCEL_EXTS and not kind:      # the workbook
                rec["summary"], rec["curves"] = R.read_inview_workbook(p)
            elif kind.lower() == "results":
                rec["summary"] = R.read_inview_table(p)
            elif re.fullmatch(r"test[\s_-]*\d+", kind, re.I):
                rec["curves"][int(re.search(r"\d+", kind).group())] = \
                    R.read_inview_table(p)
        except Exception as exc:
            print(f"  !! {p.name}: {type(exc).__name__}: {exc}")
    return [out[k] for k in sorted(out)]


records = run_exports()

# ---- 1. what actually came down ---------------------------------------------------
if not records:
    print(f"No downloaded exports for run {RUN_ID} in {DOWNLOAD_DIR}")
else:
    print(f"=== FILES DOWNLOADED FOR RUN {RUN_ID} ===")
    display(pd.DataFrame([{
        "test": r["test"], "format": "/".join(sorted(r["fmt"])),
        "files": len(r["files"]),
        "kB": round(sum(p.stat().st_size for p in r["files"]) / 1024, 1),
        "indent rows": 0 if r["summary"] is None else len(r["summary"]),
        "curves": len(r["curves"]),
    } for r in records]).set_index("test"))

# ---- 2. the per-indent summary of every test, with the campaign's OWN verdict ------
rows = []
for r in records:
    data = r["summary"]
    if data is None or data.empty or "Test" not in data:
        continue
    data = data[pd.to_numeric(data["Test"], errors="coerce").notna()]
    ok, why = R.validate_rows(R.normalize_result_columns(data.copy()),
                              expected_indents=len(data))
    for _, row in data.iterrows():
        rows.append({
            "test": r["test"],
            **{c: pd.to_numeric(row.get(c), errors="coerce") for c in
               ("Pmax", "hmax", "Hardness", "Modulus", "Stiffness^2/Load")},
            "used by campaign": "yes" if ok else "NO",
            "why not": "" if ok else why.split(" -- ")[0][:70]})
if rows:
    print(f"\n=== RESULTS CONTENT ({len(rows)} indent row(s)) ===")
    display(pd.DataFrame(rows).round(3))

# ---- 3. the raw curves ------------------------------------------------------------
if SHOW_CURVES:
    crows = []
    for r in records:
        for n, curve in sorted(r["curves"].items()):
            df = R.normalize_result_columns(curve)
            g = lambda c: (pd.to_numeric(df[c], errors="coerce") if c in df
                           else pd.Series(dtype=float))
            crows.append({"test": r["test"], "indent": n, "points": len(df),
                          "depth max (nm)": round(float(g("Contact Depth (nm)").max()), 1),
                          "load max (mN)": round(float(g("Load (mN)").max()), 2),
                          "H min": round(float(g("Hardness (GPa)").min()), 2),
                          "H max": round(float(g("Hardness (GPa)").max()), 2),
                          "E max (GPa)": round(float(g("Modulus (GPa)").max()), 1)})
    if crows:
        print("\n=== PER-INDENT CURVES ===")
        display(pd.DataFrame(crows))

# ---- 4. the tables verbatim -------------------------------------------------------
if SHOW_RAW:
    for r in records:
        for p in r["files"]:
            print(f"\n----- {p.name} -----")
            if p.suffix.lower() == ".csv":
                print(p.read_text(encoding="utf-8", errors="replace").rstrip())
            else:                  # a workbook has no verbatim text - show the grids
                for sheet, grid in pd.read_excel(p, sheet_name=None, header=None).items():
                    print(f"[sheet: {sheet}]")
                    display(grid)


## 13. 📊 Control provenance figure

Summarises who made each decision (rules, LLM, GP) across every saved campaign report.

In [ ]:
# ============ Control provenance across all Notebook 7 campaigns ============
# Reads experiment_logs/nb7_campaign_*.json and, for LLM-led runs, the matching
# campaign journals. Nothing is hand-typed; no instrument calls.
import argparse
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

HARD_ACTIONS = {"objective_met", "budget_exhausted"}
GREEN, GOLD, GREY, SLATE = "#3c7a3c", "#c9971f", "#c9ccd1", "#7a8699"


def classify(record: dict) -> str:
    action, advisor = record.get("action"), record.get("advisor")
    llm_comment = record.get("llm_comment", "")
    if action in HARD_ACTIONS:
        return "hard_stop"
    if advisor == "llm-site-planner":
        return "llm_led"
    if advisor == "llm":
        return "override"
    if advisor == "rules":
        return "concurred" if llm_comment else "unusable"
    print(f"  (unrecognised decision record, counted as 'unusable'): {record.get('advisor')!r}")
    return "unusable"


def find_logs_dir(explicit: Path) -> Path:
    if explicit.is_dir() and list(explicit.glob("nb7_campaign_*.json")):
        return explicit
    here = Path(".").resolve()
    for candidate in (here, *here.parents):
        d = candidate / "experiment_logs"
        if d.is_dir() and list(d.glob("nb7_campaign_*.json")):
            return d
    raise FileNotFoundError(f"no nb7_campaign_*.json under {explicit!r} or any parent of {here}")


def site_choices(journal: Path) -> list[dict]:
    """LLM_SITE_CHOICE events of one run (empty for rules-mode runs)."""
    if not journal.is_file():
        return []
    out = []
    for line in journal.read_text(encoding="utf-8").splitlines():
        try:
            e = json.loads(line)
        except json.JSONDecodeError:
            continue
        if e.get("event") == "LLM_SITE_CHOICE":
            out.append(e)
    return out


def load_campaigns(logs_dir: Path, journal_dir: Path) -> list[dict]:
    campaigns = []
    for f in sorted(logs_dir.glob("nb7_campaign_*.json")):
        d = json.loads(f.read_text())
        run_id = f.stem.removeprefix("nb7_campaign_")
        campaigns.append({
            "run_id": run_id,
            "n_messages": len(d.get("agent_messages", [])),
            "compiled_by": d.get("objective", {}).get("compiled_by"),
            "categories": [classify(r) for r in d.get("orchestrator_decisions", [])],
            "site_choices": site_choices(journal_dir / f"campaign_{run_id}.jsonl"),
        })
    return campaigns


def bar_with_labels(ax, x, height, colors, title):
    bars = ax.bar(x, height, color=colors, edgecolor="#333333", linewidth=0.8, width=0.62)
    top = max(max(height), 1)
    for b, v in zip(bars, height):
        ax.text(b.get_x() + b.get_width() / 2, v + top * 0.02, str(v),
                ha="center", va="bottom", fontsize=9.5, fontweight="bold")
    ax.set_title(title, fontsize=10.5, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_ylim(0, top * 1.18)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--logs", type=Path, default=Path("experiment_logs"))
    ap.add_argument("--journals", type=Path, default=None,
                    help="campaign_journals/ folder; default <repo>/notebooks/campaign_journals")
    ap.add_argument("--out", type=Path, default=None)
    args, _unknown = ap.parse_known_args()          # tolerate Jupyter's --f=...

    logs_dir = find_logs_dir(args.logs)
    repo = logs_dir.parent
    journal_dir = args.journals or repo / "notebooks" / "campaign_journals"
    out = args.out or repo / "paper" / "figures" / "fig_nb7_control_provenance"
    print(f"reading campaigns from: {logs_dir.resolve()}")
    print(f"reading journals from:  {journal_dir.resolve()}")
    print(f"writing figure to:      {out.resolve()}.{{pdf,png}}\n")
    campaigns = load_campaigns(logs_dir, journal_dir)

    # (a) orchestrator decision disposition
    disp_order = ["concurred", "override", "unusable", "llm_led", "hard_stop"]
    disp_labels = ["model\nconcurred", "model\noverrode", "advisory\nunusable",
                   "to LLM\nplanner", "hard\nstop"]
    disp_counts = Counter(cat for c in campaigns for cat in c["categories"])
    disp_values = [disp_counts.get(k, 0) for k in disp_order]
    disp_colors = [GREEN, GREY, GREY, GOLD, GREY]

    # (b) objective compilation
    n_llm = sum(c["compiled_by"] == "llm" for c in campaigns)
    n_fb = sum(c["compiled_by"] == "fallback" for c in campaigns)
    comp_labels, comp_values, comp_colors = ["LLM", "fallback\nparser"], [n_llm, n_fb], [GOLD, GREY]
    if len(campaigns) - n_llm - n_fb:
        comp_labels.append("other"); comp_values.append(len(campaigns) - n_llm - n_fb)
        comp_colors.append(GREY)

    # (c) provenance volume
    run_ids = [c["run_id"] for c in campaigns]
    msg_counts = [c["n_messages"] for c in campaigns]

    # (d) LLM site-planner decisions (LLM-led runs only)
    sc = [s for c in campaigns for s in c["site_choices"]]
    def kind(s):
        return "gp-fallback" if s.get("advisor") != "llm" else s.get("mode", "?")
    site_counts = Counter(kind(s) for s in sc)
    site_order = ["self", "consult_gp", "delegate_gp", "gp-fallback"]
    site_labels = ["LLM chose\nitself", "consulted\nthe GP", "delegated\nto the GP",
                   "GP fallback\n(no LLM)"]
    site_values = [site_counts.get(k, 0) for k in site_order]
    site_colors = [GOLD, GOLD, GOLD, GREY]
    n_llm_runs = sum(bool(c["site_choices"]) for c in campaigns)

    fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
    fig.suptitle("Where control authority sat, and what the model did with it",
                 fontsize=11.5, y=1.03)
    bar_with_labels(axes[0], disp_labels, disp_values, disp_colors,
                    f"(a) Orchestrator decisions (n={sum(disp_values)})")
    axes[0].set_ylabel("orchestrator decisions", fontsize=9)
    axes[0].tick_params(axis="x", labelsize=7.8)
    bar_with_labels(axes[1], comp_labels, comp_values, comp_colors, "(b) Objective compilation")
    axes[1].set_ylabel("campaigns", fontsize=9)
    bar_with_labels(axes[2], run_ids, msg_counts, [SLATE] * len(run_ids),
                    f"(c) Provenance volume ({len(campaigns)} campaigns)")
    axes[2].set_ylabel("logged agent messages", fontsize=9)
    axes[2].tick_params(axis="x", labelsize=6.8, rotation=55)
    for t in axes[2].texts:
        t.set_fontsize(6.5)
    for lbl in axes[2].get_xticklabels():
        lbl.set_ha("right")
    bar_with_labels(axes[3], site_labels, site_values, site_colors,
                    f"(d) Site choices, LLM-led ({sum(site_values)} rounds, {n_llm_runs} runs)")
    axes[3].set_ylabel("rounds", fontsize=9)
    axes[3].tick_params(axis="x", labelsize=7.8)

    for ax in axes:
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    fig.tight_layout()
    out.parent.mkdir(parents=True, exist_ok=True)
    for ext in ("pdf", "png"):
        fig.savefig(f"{out}.{ext}", dpi=300, bbox_inches="tight", facecolor="white")
        print("wrote", f"{out}.{ext}")
    plt.show()

    print(f"\nn = {sum(disp_values)} orchestrator decisions across {len(campaigns)} campaigns")
    print("  " + "  ".join(f"{k}={disp_counts.get(k, 0)}" for k in disp_order))
    print(f"objective compiled by LLM: {n_llm}/{len(campaigns)}")
    print(f"LLM site-planner rounds: {sum(site_values)} in {n_llm_runs} run(s): "
          + "  ".join(f"{k}={site_counts.get(k, 0)}" for k in site_order))
    print(f"total logged agent_messages: {sum(msg_counts)}")
    for c in campaigns:
        extra = (" | site modes: " + ", ".join(f"R{s['round_no']} {kind(s)}"
                                               for s in c["site_choices"])) if c["site_choices"] else ""
        print(f"  {c['run_id']}: {c['n_messages']} messages{extra}")


if __name__ == "__main__":
    main()
